# Multi-Period & Investment

T4's daily-amortized rate was a placeholder. To plan over realistic horizons — *should we build now or in five years?* — we add a **period** dimension to the model. Each period shares the same timestep structure but has independent dispatch.

You'll meet two new things:

- `periods=[...]` — an extra axis on every variable. Effect totals become *per period*, not just per timestep.
- **`Investment`** — a one-time build decision: the optimizer picks **when** (which period) and **at what size**.

In [ ]:
from datetime import datetime, timedelta

import pandas as pd

from fluxopt import Carrier, Converter, Effect, Flow, Investment, Port, Sizing, optimize

## 1. Add `periods=`

Pass a list of integer period labels — typically years. The same `timesteps` apply within each period; the same data repeats unless you broadcast over the period dimension.

We model three planning periods (2025, 2030, 2035) with five-year gaps and a representative 24-hour profile per period. Boiler size is set with plain `Sizing` for now: built immediately, paid for once.

In [ ]:
n = 24
timesteps = [datetime(2025, 1, 15) + timedelta(hours=h) for h in range(n)]
periods = [2025, 2030, 2035]
demand = [
    10,
    10,
    8,
    8,
    8,
    12,
    25,
    60,
    70,
    75,
    75,
    70,
    65,
    60,
    55,
    50,
    45,
    40,
    30,
    25,
    20,
    15,
    12,
    10,
]
peak = max(demand)

In [ ]:
def build(**extra):
    return optimize(
        timesteps=timesteps,
        periods=periods,
        carriers=[Carrier('gas'), Carrier('heat')],
        effects=[Effect('cost', unit='EUR')],
        ports=[
            Port(
                'gas_grid',
                imports=[
                    Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.08}),
                ],
            ),
            Port(
                'workshop',
                exports=[
                    Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand]),
                ],
            ),
        ],
        converters=[
            Converter.boiler(
                'boiler',
                thermal_efficiency=0.9,
                fuel_flow=Flow('gas', size=300),
                thermal_flow=Flow('heat', size=Sizing(min_size=20, max_size=200, effects_per_size={'cost': 1500})),
            ),
        ],
        objective_effects='cost',
        **extra,
    )


result = build()
result.effect_totals.drop_sel(effect='penalty').to_dataframe(name='total')

`effect_totals` is now 2-D — `effect × period`. Per-period costs are identical because the data is identical; only the **objective** weights them:

In [ ]:
# Default weights are inferred from period gaps — five-year gaps here → weight 5 each.
pd.Series([5, 5, 5], index=pd.Index(periods, name='period'), name='weight (default)')

In [ ]:
print(f'Objective: {result.objective:.2f} EUR  (= sum of weight × per-period cost)')

## 2. NPV weights

Future euros are worth less today. Discount each period to present value with a 5 % rate; physical effects (CO₂, fuel kWh) keep flat weights, so override only on cost via `period_weights=`.

In [ ]:
r = 0.05
period_offsets = [0, 5, 10]  # year offsets of each period from today
npv_weights = [round(sum(1 / (1 + r) ** (y0 + y) for y in range(5)), 3) for y0 in period_offsets]

result_npv = build(period_weights=npv_weights)

pd.DataFrame(
    {
        'flat': [5, 5, 5],
        'NPV (r=5%)': npv_weights,
        'cost/period': result_npv.effect_totals.sel(effect='cost').values.round(2),
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
pd.Series(
    {
        'Flat (default)': round(result.objective, 2),
        'NPV (r=5%)': round(result_npv.objective, 2),
    },
    name='objective (EUR)',
)

## 3. Investment timing

`Sizing` builds *immediately, once*, and the cost lands in the first period. **`Investment`** lets the optimizer pick the period and gives you separate one-time vs recurring cost slots:

- `effects_per_size_at_build` — CAPEX per MW, charged in the build period only.
- `effects_per_size_recurring` — annualized OPEX per MW, charged every active period.
- `lifetime` — how many periods the asset stays active after build (`None` = forever).
- `prior_size` — pre-existing capacity available from period 0 (covers demand before the new build comes online).

Replace the boiler's `Sizing` with `Investment` and watch the optimizer pick a period:

In [ ]:
result_inv = optimize(
    timesteps=timesteps,
    periods=periods,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port(
            'gas_grid',
            imports=[
                Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.08}),
            ],
        ),
        Port(
            'workshop',
            exports=[
                Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow('gas', size=300),
            thermal_flow=Flow(
                'heat',
                size=Investment(
                    min_size=50,
                    max_size=200,
                    mandatory=True,
                    effects_per_size_at_build={'cost': 1500},
                ),
            ),
        ),
    ],
    objective_effects='cost',
    period_weights=npv_weights,
)

build_period = result_inv.solution['invest--build'].sel(flow='boiler(heat)').to_dataframe(name='built')
size_chosen = float(result_inv.solution['invest--size'].sel(flow='boiler(heat)').values)
print(f'Optimal size: {size_chosen:.1f} kW')
build_period

With NPV-discounted weights, the optimizer would *delay* CAPEX if it could — but `mandatory=True` plus immediate demand forces a build in 2025. Try `mandatory=False` with a `prior_size`, or add a fuel-price ramp across periods, and the build period shifts.

## Recap

`periods=[...]` adds a period axis to every variable; `period_weights` (global or per-`Effect`) controls how much each period contributes to the objective. `Investment` replaces `Sizing` when you need build *timing*: separate one-time CAPEX from recurring OPEX, and let the optimizer pick the period.

That's the full tutorial sequence. Browse the [examples](examples/heat-system.ipynb) for fully-worked scenarios that combine these features in realistic systems, or jump to the [API reference](../api/) to see what else each object can do.